In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

# Set random seed for reproducibility
np.random.seed(42)

# 1. Load the games dimension table created in the previous step
try:
    df_games = pd.read_csv("dim_games.csv")
    date_columns = ["announcement_date", "preorder_date", "early_access_date", "release_date"]
    for col in date_columns:
        df_games[col] = pd.to_datetime(df_games[col])
except FileNotFoundError:
    raise FileNotFoundError("dim_games.csv not found! Please run the previous script first.")

# 2. Generate a continuous daily date range from 2020-01-01 to 2025-12-31
start_date = "2020-01-01"
end_date = "2025-12-31"
dates = pd.date_range(start=start_date, end=end_date, freq="D")

# Create a DataFrame for console sales
console_sales_records = []
# Create a DataFrame for subscriptions
subscription_records = []

# Initializing subscriber base (starting around 4M in Jan 2020, growing to ~34M by 2025)
current_subscribers = 4000000 

print("Generating daily dynamic time-series data...")

for current_date in dates:
    # --- CONSOLE SALES MODELING ---
    # Before Xbox Series X|S launch (Nov 2020), sales are 0 for Series X/S
    if current_date < pd.to_datetime("2020-11-10"):
        sales_series_x = 0
        sales_series_s = 0
    else:
        # Base daily sales with weekly seasonality (higher on Fridays/Saturdays)
        weekday = current_date.weekday()
        weekend_multiplier = 1.4 if weekday >= 4 else 0.8
        
        # Annual seasonality (Holiday spike in November/December)
        month = current_date.month
        holiday_multiplier = 2.5 if month in [11, 12] else (1.2 if month in [6, 7] else 1.0)
        
        # Check if any major game released today or in the last 3 days (halo effect)
        recent_releases = df_games[
            (df_games["release_date"] >= current_date - timedelta(days=3)) & 
            (df_games["release_date"] <= current_date)
        ]
        
        event_multiplier = 1.0
        if not recent_releases.empty:
            # If it's a Day One Game Pass or Major Third-Party release, hardware sales surge
            if any(recent_releases["category"] == "Day One Game Pass"):
                event_multiplier = 3.2
            elif any(recent_releases["category"] == "Third-Party Control Group"):
                event_multiplier = 2.8
            else:
                event_multiplier = 1.8

        # Base generation with Gaussian noise to avoid flat lines
        base_x = np.random.normal(1200, 200)
        base_s = np.random.normal(1800, 300) # Series S typically sold slightly more units due to lower price point
        
        sales_series_x = max(200, int(base_x * weekend_multiplier * holiday_multiplier * event_multiplier))
        sales_series_s = max(300, int(base_s * weekend_multiplier * holiday_multiplier * event_multiplier))

    console_sales_records.append({
        "date": current_date.strftime("%Y-%m-%d"),
        "console_model": "Xbox Series X",
        "units_sold": sales_series_x
    })
    console_sales_records.append({
        "date": current_date.strftime("%Y-%m-%d"),
        "console_model": "Xbox Series S",
        "units_sold": sales_series_s
    })

    # --- SUBSCRIPTION MODELING (XBOX GAME PASS ULTIMATE) ---
    # Normal daily churn and organic acquisition
    natural_new = int(np.random.normal(15000, 3500))
    natural_churn = int(np.random.normal(12000, 2800))
    
    # Check for Day One release impact on subscriptions
    day_one_releases = df_games[
        (df_games["release_date"] == current_date) & 
        (df_games["category"] == "Day One Game Pass")
    ]
    
    early_access_releases = df_games[
        (df_games["early_access_date"] == current_date) & 
        (df_games["category"] == "Day One Game Pass")
    ]

    campaign_boost_new = 0
    if not day_one_releases.empty:
        # Massive spike for Day One releases (e.g., Starfield, CoD BO6)
        game_title = day_one_releases.iloc[0]["game_title"]
        if "Call of Duty" in game_title or "Starfield" in game_title:
            campaign_boost_new = int(np.random.normal(450000, 80000)) # Blockbuster spike
        else:
            campaign_boost_new = int(np.random.normal(150000, 30000)) # Standard Day One spike
            
    elif not early_access_releases.empty:
        # Early access spike (e.g., Forza Horizon 5, Starfield Premium Edition)
        campaign_boost_new = int(np.random.normal(120000, 25000))

    total_new_subs = max(1000, natural_new + campaign_boost_new)
    total_churn_subs = max(500, natural_churn)
    
    # Update active subscriber pool
    current_subscribers = current_subscribers + total_new_subs - total_churn_subs

    subscription_records.append({
        "date": current_date.strftime("%Y-%m-%d"),
        "active_subscribers": max(3000000, current_subscribers),
        "new_subscriptions": total_new_subs,
        "churned_subscriptions": total_churn_subs
    })

# Convert records to DataFrames
df_console_sales = pd.DataFrame(console_sales_records)
df_subscriptions = pd.DataFrame(subscription_records)

# Save to CSV files
df_console_sales.to_csv("fact_console_sales.csv", index=False)
df_subscriptions.to_csv("fact_subscriptions.csv", index=False)

print("ETL Data Generation Completed Successfully!")
print(f"Generated {len(df_console_sales)} rows for console sales.")
print(f"Generated {len(df_subscriptions)} rows for subscription metrics.")

Generating daily dynamic time-series data...
ETL Data Generation Completed Successfully!
Generated 4384 rows for console sales.
Generated 2192 rows for subscription metrics.
